In [1]:
# 203. Sort the feature data by time and perform a chronological train/test split: earlier periods train, later periods test. Never a random split.

import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="nopis"
)

query = """
SELECT
    grid_id,
    feature_timestamp,
    avg_activity,
    activity_growth,
    active_hours,
    peak_ratio,
    variability,
    internet_share
FROM network_feature_table
ORDER BY feature_timestamp, grid_id;
"""

ml_df = pd.read_sql(query, conn)

ml_df["feature_timestamp"] = pd.to_datetime(
    ml_df["feature_timestamp"]
)

ml_df.head()

D:\NOPIS\tmp\ipykernel_35392\954284187.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ml_df = pd.read_sql(query, conn)


,grid_id,feature_timestamp,avg_activity,activity_growth,active_hours,peak_ratio,variability,internet_share
0,1,2013-10-31 18:30:00,NaN,NaN,NaN,NaN,NaN,0.937815
1,2,2013-10-31 18:30:00,NaN,NaN,NaN,NaN,NaN,0.937878
2,3,2013-10-31 18:30:00,NaN,NaN,NaN,NaN,NaN,0.937944
3,4,2013-10-31 18:30:00,NaN,NaN,NaN,NaN,NaN,0.937633
4,5,2013-10-31 18:30:00,NaN,NaN,NaN,NaN,NaN,0.937811


In [2]:
# 203. Create the future t+1 target for the chronological ML dataset.

activity_query = """
SELECT
    f.grid_id,
    d.timestamp AS activity_timestamp,
    f.total_activity
FROM fact_network_activity f
JOIN dim_time d
    ON f.time_key = d.time_key
ORDER BY f.grid_id, d.timestamp;
"""

activity_df = pd.read_sql(activity_query, conn)

activity_df["activity_timestamp"] = pd.to_datetime(
    activity_df["activity_timestamp"]
)

# Target threshold selected in ML1
threshold = 1154.5988

# The feature timestamp t must match the activity timestamp t+1
activity_df["feature_timestamp"] = (
    activity_df["activity_timestamp"] - pd.Timedelta(hours=1)
)

activity_df["target"] = (
    activity_df["total_activity"] > threshold
).astype(int)

ml_df = ml_df.merge(
    activity_df[
        ["grid_id", "feature_timestamp", "target"]
    ],
    on=["grid_id", "feature_timestamp"],
    how="left"
)

ml_df.head()

D:\NOPIS\tmp\ipykernel_35392\2458390181.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  activity_df = pd.read_sql(activity_query, conn)


,grid_id,feature_timestamp,avg_activity,activity_growth,active_hours,peak_ratio,variability,internet_share,target
0,1,2013-10-31 18:30:00,NaN,NaN,NaN,NaN,NaN,0.937815,0.0
1,2,2013-10-31 18:30:00,NaN,NaN,NaN,NaN,NaN,0.937878,0.0
2,3,2013-10-31 18:30:00,NaN,NaN,NaN,NaN,NaN,0.937944,0.0
3,4,2013-10-31 18:30:00,NaN,NaN,NaN,NaN,NaN,0.937633,0.0
4,5,2013-10-31 18:30:00,NaN,NaN,NaN,NaN,NaN,0.937811,0.0


In [3]:
# 203. Prepare the dataset by removing rows without complete features or a t+1 target.

feature_columns = [
    "avg_activity",
    "activity_growth",
    "active_hours",
    "peak_ratio",
    "variability",
    "internet_share"
]

ml_df = ml_df.dropna(
    subset=feature_columns + ["target"]
).copy()

ml_df = ml_df.sort_values(
    "feature_timestamp"
).reset_index(drop=True)

print("Rows available for ML:", len(ml_df))
print("First timestamp:", ml_df["feature_timestamp"].min())
print("Last timestamp:", ml_df["feature_timestamp"].max())

Rows available for ML: 1036315
First timestamp: 2013-11-02 17:30:00
Last timestamp: 2013-11-07 16:30:00


In [4]:
# 203. Sort the feature data by time and perform a chronological train/test split: earlier periods train, later periods test. Never a random split.

unique_times = sorted(
    ml_df["feature_timestamp"].unique()
)

split_index = int(len(unique_times) * 0.80)

train_times = unique_times[:split_index]
test_times = unique_times[split_index:]

train_df = ml_df[
    ml_df["feature_timestamp"].isin(train_times)
].copy()

test_df = ml_df[
    ml_df["feature_timestamp"].isin(test_times)
].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

Training rows: 825614
Testing rows: 210701


In [5]:
# 204. Record and report the earliest and latest timestamp in each of the train and test sets.

print("TRAIN SET")
print("Earliest:", train_df["feature_timestamp"].min())
print("Latest:  ", train_df["feature_timestamp"].max())

print("\nTEST SET")
print("Earliest:", test_df["feature_timestamp"].min())
print("Latest:  ", test_df["feature_timestamp"].max())

TRAIN SET
Earliest: 2013-11-02 17:30:00
Latest:   2013-11-06 16:30:00

TEST SET
Earliest: 2013-11-06 17:30:00
Latest:   2013-11-07 16:30:00


In [6]:
# 204. Verify that train and test time periods do not overlap.

assert train_df["feature_timestamp"].max() < test_df["feature_timestamp"].min()

print("PASS: Train and test periods do not overlap.")

PASS: Train and test periods do not overlap.


In [7]:
# 205. Train Logistic Regression or a Decision Tree.

from sklearn.linear_model import LogisticRegression

feature_columns = [
    "avg_activity",
    "activity_growth",
    "active_hours",
    "peak_ratio",
    "variability",
    "internet_share"
]

X_train = train_df[feature_columns]
y_train = train_df["target"]

X_test = test_df[feature_columns]
y_test = test_df["target"]

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

Training features: (825614, 6)
Testing features: (210701, 6)


In [8]:
# 205. Train Logistic Regression or a Decision Tree.

model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [9]:
# 206. Evaluate accuracy plus precision, recall and the class balance. Report the base rate — the proportion of positive labels — alongside accuracy, because accuracy is meaningless without it.

from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# Make predictions on the unseen test data
y_pred = model.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

# Class balance / base rate
positive_count = y_test.sum()
negative_count = (y_test == 0).sum()
base_rate = y_test.mean()

print("TEST SET EVALUATION")
print("-------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")

print("\nCLASS BALANCE")
print("-------------")
print(f"Negative cases: {negative_count:,}")
print(f"Positive cases: {positive_count:,}")
print(f"Base rate     : {base_rate:.4f} ({base_rate * 100:.2f}%)")

TEST SET EVALUATION
-------------------
Accuracy : 0.9632
Precision: 0.8784
Recall   : 0.8067

CLASS BALANCE
-------------
Negative cases: 185,309
Positive cases: 25,392.0
Base rate     : 0.1205 (12.05%)


Precision 87.84% → when the model flags a risk, about 88% of those flags are actually positive according to our proxy label.  
Recall 80.67% → the model catches about 81% of the positive cases.  
Base rate 12.05% → only about 12% of test cases are positive.

In [10]:
# 206. Inspect the confusion matrix to investigate the high accuracy.

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[182473   2836]
 [  4908  20484]]


                 Predicted
                 0       1
Actual 0            182473   2836  
Actual 1              4908  20484

In [11]:
# 207. Inspect the coefficients or the feature importances and check they are operationally plausible.

coefficients = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": model.coef_[0]
})

coefficients["direction"] = coefficients["coefficient"].apply(
    lambda x: "Positive risk" if x > 0 else "Negative risk"
)

coefficients.sort_values(
    "coefficient",
    ascending=False
)

,feature,coefficient,direction
3,peak_ratio,2.062651,Positive risk
1,activity_growth,0.520892,Positive risk
0,avg_activity,0.008062,Positive risk
4,variability,-0.006849,Negative risk
2,active_hours,-0.027760,Negative risk
5,internet_share,-11.348133,Negative risk


In [12]:
# 207. Display the Logistic Regression coefficients.

print(coefficients.to_string(index=False))

        feature  coefficient     direction
   avg_activity     0.008062 Positive risk
activity_growth     0.520892 Positive risk
   active_hours    -0.027760 Negative risk
     peak_ratio     2.062651 Positive risk
    variability    -0.006849 Negative risk
 internet_share   -11.348133 Negative risk


avg_activity +0.0081 → higher recent activity increases predicted risk. Plausible.

activity_growth +0.5209 → increasing activity increases predicted risk. Plausible.

active_hours −0.0278 → more consistently active hours slightly reduce predicted risk. Less intuitive, but possible given the other features.

peak_ratio +2.0627 → stronger peaks increase predicted risk. Plausible.

variability −0.0068 → higher variation slightly reduces predicted risk. Not obviously expected, so this should be noted rather than over-interpreted.

internet_share −11.3481 → higher internet share strongly reduces predicted risk. Needs caution, because this is a strong relationship learned from this dataset and proxy label, not proof of a real network relationship.

In [13]:
# 208. Find the available NP3 alert table.

tables = pd.read_sql("SHOW TABLES FROM nopis", conn)

print(tables.to_string(index=False))

      Tables_in_nopis
             dim_grid
             dim_time
fact_network_activity
        grid_features
network_feature_table


D:\NOPIS\tmp\ipykernel_35392\128475046.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables = pd.read_sql("SHOW TABLES FROM nopis", conn)


In [14]:
# 208. Compare the predictions against the rule-based NP3 alerts and characterize where they disagree.

# Create a copy of the test data
comparison_df = test_df[
    ["grid_id", "feature_timestamp"]
].copy()

comparison_df["ml_prediction"] = y_pred

# NP3 HIGH_ACTIVITY rule
# current activity at t+1 compared with the within-day baseline
comparison_df = comparison_df.merge(
    activity_df[
        ["grid_id", "feature_timestamp", "total_activity"]
    ],
    on=["grid_id", "feature_timestamp"],
    how="left"
)

# Calculate the NP3-style baseline for each grid/day
activity_df["date"] = activity_df["activity_timestamp"].dt.date

daily_baseline = (
    activity_df
    .groupby(["grid_id", "date"])["total_activity"]
    .transform("median")
)

activity_df["baseline_activity"] = daily_baseline

# Map the baseline to the comparison data
comparison_df = comparison_df.merge(
    activity_df[
        ["grid_id", "feature_timestamp", "baseline_activity"]
    ],
    on=["grid_id", "feature_timestamp"],
    how="left"
)

# Same HIGH_ACTIVITY rule used by NP3
comparison_df["np3_alert"] = (
    comparison_df["total_activity"]
    >= comparison_df["baseline_activity"] * 1.50
).astype(int)

print(comparison_df.head())

   grid_id   feature_timestamp  ml_prediction  total_activity  \
0     6695 2013-11-06 17:30:00            0.0        622.6222   
1     6696 2013-11-06 17:30:00            0.0        248.3977   
2     6702 2013-11-06 17:30:00            0.0         39.0102   
3     6676 2013-11-06 17:30:00            0.0        416.7610   
4     6703 2013-11-06 17:30:00            0.0         60.5482   

   baseline_activity  np3_alert  
0          830.40105          0  
1          470.45120          0  
2           68.04340          0  
3          967.28285          0  
4          102.50740          0  


In [15]:
# 208. Compare the ML predictions with NP3 HIGH_ACTIVITY alerts.

comparison_df["comparison"] = "AGREE"

comparison_df.loc[
    (comparison_df["ml_prediction"] == 1) &
    (comparison_df["np3_alert"] == 0),
    "comparison"
] = "ML only"

comparison_df.loc[
    (comparison_df["ml_prediction"] == 0) &
    (comparison_df["np3_alert"] == 1),
    "comparison"
] = "NP3 only"

print(comparison_df["comparison"].value_counts())

comparison
AGREE       184306
ML only      22817
NP3 only      3578
Name: count, dtype: int64


In [16]:
# 208. Calculate the percentage of agreements and disagreements between ML and NP3.

comparison_counts = comparison_df["comparison"].value_counts()

print(
    (comparison_counts / len(comparison_df) * 100)
    .round(2)
    .astype(str)
    + "%"
)

comparison
AGREE       87.47%
ML only     10.83%
NP3 only      1.7%
Name: count, dtype: str


In [17]:
# 208. Display examples where the ML model and NP3 rule disagree.

disagreements = comparison_df[
    comparison_df["comparison"] != "AGREE"
]

print(disagreements.head(10))

     grid_id   feature_timestamp  ml_prediction  total_activity  \
14      6675 2013-11-06 17:30:00            1.0       1753.1347   
42      6747 2013-11-06 17:30:00            1.0       1457.4577   
46      6751 2013-11-06 17:30:00            1.0        783.7082   
47      6752 2013-11-06 17:30:00            1.0       1127.8235   
49      6753 2013-11-06 17:30:00            1.0       1769.1570   
70      6674 2013-11-06 17:30:00            1.0       2371.9817   
72      6672 2013-11-06 17:30:00            1.0       2578.9274   
107     6673 2013-11-06 17:30:00            1.0       1715.1144   
110     6658 2013-11-06 17:30:00            1.0       1738.2828   
111     6659 2013-11-06 17:30:00            1.0       1702.5734   

     baseline_activity  np3_alert comparison  
14          3008.64700          0    ML only  
42          1733.49155          0    ML only  
46          1681.83535          0    ML only  
47          1844.96665          0    ML only  
49          2268.15065     

### 209. Write three observations about where the model adds value and where it does not.

1. The ML model adds value by identifying additional risk cases that the NP3 rule does not flag. The ML-only cases were 10.83% of the test data.

2. The ML model and NP3 rule agree on most cases, with 87.47% agreement. This shows that the model provides a similar signal to the existing rule in many cases.

3. The ML model is not a replacement for the rule. NP3 still identified 1.70% of cases that the ML model missed, so both approaches have limitations and should be used as investigation signals rather than proof of congestion.


In [20]:
# 217. Save the trained ML3 model for FastAPI model serving.

import joblib
from pathlib import Path

model_path = Path(r"D:\NOPIS\ml\ml3_logistic_regression.joblib")

joblib.dump(model, model_path)

print("Model saved successfully.")
print("Model path:", model_path)
print("Model type:", type(model).__name__)

Model saved successfully.
Model path: D:\NOPIS\ml\ml3_logistic_regression.joblib
Model type: LogisticRegression


In [21]:
# Verify that the saved model can be loaded successfully.

loaded_model = joblib.load(model_path)

print("Model loaded successfully.")
print("Model type:", type(loaded_model).__name__)

Model loaded successfully.
Model type: LogisticRegression


In [23]:
# Save ML3 model metadata

import json
from pathlib import Path

metadata = {
    "model_version": "ml3_v1.0",
    "model_type": "LogisticRegression",
    "features": [
        "avg_activity",
        "activity_growth",
        "active_hours",
        "peak_ratio",
        "variability",
        "internet_share"
    ],
    "target": "next_hour_high_activity",
    "target_threshold": 1154.5988
}

metadata_path = Path(r"D:\NOPIS\ml\ml3_model_metadata.json")

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("Metadata saved successfully.")
print("Metadata path:", metadata_path)

print("\nMetadata:")
print(json.dumps(metadata, indent=4))

Metadata saved successfully.
Metadata path: D:\NOPIS\ml\ml3_model_metadata.json

Metadata:
{
    "model_version": "ml3_v1.0",
    "model_type": "LogisticRegression",
    "features": [
        "avg_activity",
        "activity_growth",
        "active_hours",
        "peak_ratio",
        "variability",
        "internet_share"
    ],
    "target": "next_hour_high_activity",
    "target_threshold": 1154.5988
}
